# 12 · Frozen billed-value model explainability

**Primary Phase 09 research · Fieldwise Cloud fictional development evidence**

The unit is a canonical customer account at an observation. The approved
target is six calendar months of undiscounted gross invoice issuance as
recorded. Earnings, margin, collections and full lifetime value have
different definitions. This study describes behaviour and errors of the
frozen predictor on the previously evaluated development validation set.

The notebook renders checksum-verified results on the **final clipped
GBP scale**. Compatibility checks identify which raw explanations can be
interpreted. All reporting is internal. The notebook performs no fitting,
explanation generation or reserve inference.
[Phase 09 scope](../../docs/05_ml/phase09_explainability_scope_and_validation.md)
and [approved economic definition](../../docs/05_ml/phase07_clv_economic_definition_and_validation.md).

In [ ]:
from os import getenv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter

from nowa_customer_intelligence.explainability.research import WORKSPACE, load_report

model = "clv"
root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()
)
requested = getenv("NCI_XAI_WORKSPACE")
workspace = Path(requested) if requested else root / WORKSPACE
r = load_report(workspace, model) if requested or workspace.exists() else None
plt.rcParams.update(
    {
        "figure.figsize": (10, 4.4),
        "figure.dpi": 125,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.20,
        "axes.axisbelow": True,
    }
)
blue, warm, grey = "#246782", "#a44d28", "#65757b"
cadence_colours = {"monthly": blue, "annual": warm, "mixed": "#694b85"}
footer = "Fieldwise fictional development · internal research · six-calendar-month gross issuance"
units = "Probability" if model == "churn" else "Gross issued GBP"


def show(fig):
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.text(0.015, 0.015, footer, fontsize=8, color=grey)
    plt.show()
    plt.close(fig)


def readable(name):
    return name.replace("_", " ")

## Research dashboard and frozen evidence

The accepted design and model checksums precede unpickling in the research
runner. Input populations stay anchored to the accepted development evidence.
The dashboard distinguishes row counts from independent account counts.
Positive outcome support matters when interpreting classification error.
Clustering and protected reserves are outside this report.

In [ ]:
if r:
    population = pd.DataFrame(r["reference_populations"]).T
    display(population[["rows", "accounts", "cadence", "month"]])
    display(pd.Series(r["model"], name="Frozen model evidence").to_frame())
    display(pd.Series(r["accepted_metrics"], name="Retained validation evidence").to_frame())
    display(pd.Series(r["environment"]["packages"], name="Research environment").to_frame())
    display(
        Markdown(
            f"**Primary explanation** · {r['primary']['method']} · {r['primary']['scale']}. "
            f"**Protocol SHA-256** · `{r['protocol_sha256']}`."
        )
    )
else:
    display(
        Markdown(
            "A verified Phase 09 research report is required for empirical results. "
            "The notebook README gives the isolated replay command."
        )
    )

## Preprocessing, feature dictionary and positional lineage

**Frozen raw registry → fitted numerical imputer and categorical encoder →
fitted selection where present → frozen tree estimator → output contract.**
The original field table records provenance and missingness conventions.
The transformation table follows physical positions, including separate
dummies and indicators. Collapsing a fitted transformed explanation uses
a signed sum of coordinates that originated in the same field.

For example, a missingness flag and the imputed numerical coordinate can
represent one original measure. Signed contributions +0.06 and −0.02 yield
+0.04 under that mapping. A separate game on the raw feature blocks uses
a different set of coalitions.

In [ ]:
if r:
    dictionary = pd.DataFrame(r["feature_dictionary"])
    lineage = pd.DataFrame(r["lineage"])
    widths = r["compatibility"]["prediction_contract"]["widths"]
    with pd.option_context("display.max_rows", 55, "display.max_colwidth", 85):
        display(dictionary)
        display(lineage.assign(position=np.arange(len(lineage)) + 1))
    fig, ax = plt.subplots(figsize=(8, 3.6))
    names = ["Original fields", "Transformed coordinates", "Estimator inputs"]
    bars = ax.barh(names, widths, color=[grey, warm, blue])
    ax.bar_label(bars, padding=5)
    ax.set(
        title="Frozen feature lineage · complete fitted pipeline",
        xlabel="Dimension count",
        xlim=(0, max(widths) * 1.16),
    )
    show(fig)

## Billed-value output, clipping and historical errors

The fitted model outputs $\hat{y}_{\mathrm{raw}}$ in signed issued
GBP. The accepted scoring wrapper is
$\hat{y}_{\mathrm{GBP}}=\max(0,\hat{y}_{\mathrm{raw}})$.
Applying a zero floor separately to a base value or feature
contributions changes the decomposition. The actual game below
evaluates the floor at every model call.

Prediction error is forecast minus mature issuance. The original
validation MAE is the average absolute error across 355 rows.
RMSE weights large misses more heavily. A reconciled explanation
can accompany a large signed error.

In [ ]:
if r:
    dist = pd.DataFrame(r["distribution"])
    contract = r["compatibility"]["prediction_contract"]
    display(pd.Series(contract, name="Validation prediction audit").to_frame())
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
    if model == "churn":
        threshold = r["model"]["threshold"]
        edges = np.linspace(0, 1, 26)
        for actual, plot_colour in ((0, blue), (1, warm)):
            frame = dist.loc[dist.target.eq(actual), "prediction"]
            axes[0].hist(
                frame,
                bins=edges,
                histtype="step",
                lw=2,
                color=plot_colour,
                label=f"Outcome {actual} · {len(frame)} rows",
            )
        axes[0].axvline(threshold, ls="--", color=grey, label=f"Threshold {threshold:.4f}")
        axes[0].set(
            title="Development validation score distribution",
            xlabel="Class-1 probability",
            ylabel="Observations",
            yscale="log",
        )
        axes[0].legend(fontsize=8)
        m = r["accepted_metrics"]
        counts = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
        image = axes[1].imshow(counts, cmap="Blues")
        for (row, col), value in np.ndenumerate(counts):
            axes[1].text(
                col,
                row,
                str(value),
                ha="center",
                va="center",
                color="white" if value > counts.max() / 2 else "black",
            )
        axes[1].set(
            title="Frozen threshold · 717 validation rows",
            xlabel="Threshold decision",
            ylabel="Recorded outcome",
            xticks=[0, 1],
            xticklabels=["No alert", "Alert"],
            yticks=[0, 1],
            yticklabels=["Negative", "Positive"],
        )
        axes[1].grid(False)
        fig.colorbar(image, ax=axes[1], label="Observations")
    else:
        for name, group in dist.groupby("cadence"):
            axes[0].scatter(
                group.target,
                group.prediction,
                s=18,
                alpha=0.46,
                color=cadence_colours[name],
                label=f"{name} · {len(group)}",
            )
        boundary = max(dist.target.max(), dist.prediction.max())
        axes[0].plot([0, boundary], [0, boundary], ls="--", color=grey, label="Perfect agreement")
        axes[0].set(
            title="Observed issuance and frozen forecast",
            xlabel="Actual gross issued GBP",
            ylabel="Clipped prediction GBP",
        )
        axes[0].legend(fontsize=8)
        axes[1].hist(dist.error, bins=25, color=blue)
        axes[1].axvline(0, ls="--", color=grey)
        axes[1].set(
            title="Signed prediction errors · 355 validation rows",
            xlabel="Prediction minus actual GBP",
            ylabel="Observations",
        )
    show(fig)

if r and model == "clv":
    negative = dist.loc[dist.raw.lt(0), ["cadence", "raw", "prediction"]]
    display(
        negative.rename(columns={"raw": "Pre-clip GBP", "prediction": "Final GBP"}).reset_index(
            drop=True
        )
    )
    display(
        Markdown(
            f"Clipping changed **{len(negative)}** validation predictions. "
            "Each became zero under the accepted output rule."
        )
    )

## Training-only background and explanation observations

The reference uses up to 64 distinct training accounts. Each contributes
one latest eligible row, with a stable episode tie-break where needed.
Proportionate cadence quotas and seed 909 fix the background. The 96
global explanation rows are account-balanced on the previously evaluated
validation population, with a fixed seed and cadence quotas. Local roles
are selected separately and only the global sample enters global means.

Using the latest fitting row per account concentrates the background near
the end of training. This is a defined reference distribution. Compare
against alternatives within the same authorised training population.

In [ ]:
if r:
    background = r["background"]
    display(pd.Series(background, name="Reference background").to_frame())
    cadence_mix = pd.DataFrame(
        {
            "Fitting reference": background["cadence"],
            "Explained account sample": r["reference_populations"]["explained_global"]["cadence"],
        }
    ).fillna(0)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    cadence_mix.div(cadence_mix.sum()).T.plot.bar(
        stacked=True,
        rot=0,
        color=cadence_colours,
        ax=axes[0],
        title="Cadence composition · background 64, explained 96",
        ylabel="Share of accounts",
    )
    axes[0].yaxis.set_major_formatter(PercentFormatter(1))
    pd.Series(background["month"]).plot.bar(
        color=blue,
        ax=axes[1],
        title="Latest eligible fitting rows by month",
        ylabel="Distinct reference accounts",
        xlabel="Observation month",
    )
    axes[1].tick_params(axis="x", labelrotation=20)
    show(fig)

## Additive attribution and compatibility gates

Let $G$ be the four blocks in the frozen registry and $B$ the 64
equally weighted training reference rows. For a case $x$ and
coalition $S\subseteq G$, construct $z(x,b,S)$ in the original feature
space. Keep the complete blocks in $S$ from $x$ and fill all remaining
blocks from the same background row $b$. Apply the fitted preprocessing
and regressor to each hybrid and clip each prediction before averaging.

$$
f(z)=\max(0,h(z)),\qquad
v_x(S)=\frac{1}{64}\sum_{b\in B}f(z(x,b,S)).
$$

Enumerate all 16 coalitions. A block's Shapley value averages its
marginal contribution over the 24 possible insertion orders.

$$
\phi_j(x)=\sum_{S\subseteq G\setminus\{j\}}
  \frac{v_x(S\cup\{j\})-v_x(S)}{4\,\binom{3}{|S|}},
\qquad f(x)=v_x(\varnothing)+\sum_{j\in G}\phi_j(x).
$$

The baseline is the mean clipped forecast on the selected training
accounts. The block design preserves correlations within borrowed
blocks. Hybrids may still break relationships across blocks, and the
game cannot isolate individual fields' contributions.

The interventional TreeExplainer check failed on all 12 signed raw
predictions, with a maximum discrepancy of £9.66. The cause remains
unresolved. The failing check took place before clipping.
Path-dependent TreeSHAP reconciled on the signed pre-clip output,
with a different training-path reference. The primary analysis uses
exact enumeration over four prespecified raw feature blocks and
the final clipped GBP callable. Global and local charts identify
these groups explicitly. No per-feature TreeSHAP beeswarm is
presented for CLV. Model-agnostic signed distributions describe the
valid finite-background block game.

In [ ]:
if r:
    compatibility = pd.DataFrame(r["compatibility"]["tree"]).T
    display(compatibility)
    primary = r["primary"]
    checks = {
        "Primary final output": primary["reconciliation"],
        "Finite-background block game": r["grouped_game"]["reconciliation"],
    }
    checks.update(
        {
            f"Tree {name}": evidence["original"]["reconciliation"]
            for name, evidence in r["tree_explanations"].items()
            if "original" in evidence
        }
    )
    display(pd.DataFrame(checks).T)
    residual = np.asarray(primary["base"]) + np.asarray(primary["values"]).sum(axis=1)
    residual -= np.asarray(primary["predictions"])
    fig, ax = plt.subplots()
    ax.scatter(range(1, len(residual) + 1), residual, s=20, color=blue, alpha=0.72)
    ax.axhline(0, color=grey, ls="--")
    ax.set(
        title=f"Signed additivity residual · {len(residual)} explained rows",
        xlabel="Global sample followed by additional local cases",
        ylabel=f"Reconstructed minus exact · {primary['scale']}",
    )
    show(fig)

## Global permutation importance for clipped predictions

The unchanged reference is the same 355 January and February 2023
development-validation observations across 180 accounts used for the
accepted £58.27 row-weighted MAE. An equal-account sensitivity starts
at £57.47 by averaging each account's MAE before averaging accounts.

Keep mature outcomes fixed. Within each set of accounts with identical
ordered observation dates, shuffle donor accounts and exchange their
entire trajectories for the selected block. Keep its columns together
and run the complete frozen predictor, including clipping. The result
for each repeat is the permuted mean absolute error minus the baseline
MAE. The summary uses 30 repeats at seed 912 and the sample standard
deviation (29 degrees of freedom). Calendar is fixed within matched
strata, so the permutation does not measure calendar reliance.

The increase is a prediction-error change in GBP, separate from the
attribution of a prediction. A joint swap preserves relationships
within the block but can create unusual combinations with unchanged
fields. The four singleton stress tests can also break linked invoice
relationships.

In [ ]:
if r:
    perm = r["permutation"]
    rows = pd.DataFrame(perm["groups"]).T
    rows["Mean account-weighted change"] = rows.account_balanced_delta.map(np.mean)
    display(
        rows[
            [
                "mean",
                "sd",
                "p05_p95",
                "Mean account-weighted change",
                "movable_accounts",
                "mean_changed_rows",
                "supported",
            ]
        ]
    )
    display(
        pd.Series(
            {"Row MAE": perm["row_mae"], "Account-weighted MAE": perm["account_balanced_mae"]},
            name="Unperturbed GBP",
        ).to_frame()
    )

    sections = [
        ("Coherent blocks", rows.iloc[:4]),
        ("Singleton stress tests", rows.iloc[4:]),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(13.4, 4.6))
    for (title, region), ax in zip(sections, axes, strict=True):
        supported = region.loc[region.supported]
        ax.boxplot(
            supported.delta_mae.tolist(),
            tick_labels=[readable(x) for x in supported.index],
            orientation="horizontal",
            patch_artist=True,
            boxprops={"facecolor": "#c9dce4"},
            medianprops={"color": blue},
        )
        ax.axvline(0, color=grey, ls="--")
        ax.set(title=title, xlabel="Increase in clipped validation MAE (GBP)")
    fig.suptitle("Matched account permutations · 355 validation observations, 30 repeats")
    show(fig)
    display(
        Markdown(
            f"The invoice-history and cadence block increased MAE by "
            f"**£{rows.loc['Invoice history and cadence', 'mean']:,.2f}** on average. "
            "It moved a set of correlated inputs, so this is joint model reliance."
        )
    )

## Global signed contributions and cadence

The separate finite-background game attributes the final clipped GBP
estimate relative to the mean prediction for the fitting reference.
Mean absolute contributions summarise how much each block matters on
these examples. Means and signed distributions by cadence reveal
whether the direction changes in different supported observations.
Contributions can be negative even when the final prediction is
constrained to be non-negative.

In [ ]:
if r:
    n_global = r["global_rows"]
    ranked = pd.Series(r["global"]["mean_absolute"]).sort_values(ascending=False)
    block = pd.Series(r["grouped_global"]["mean_absolute"]).sort_values(ascending=True)
    display(ranked.rename("Mean absolute contribution").to_frame())
    fig, axes = plt.subplots(1, 2, figsize=(13.3, 5))
    shown = ranked.head(9).sort_values()
    axes[0].barh([readable(x) for x in shown.index], shown, color=blue)
    axes[0].set(title="Primary attribution · leading coordinates", xlabel=primary["scale"])
    axes[1].barh(block.index, block, color=warm)
    axes[1].set(title="Separate grouped callable game", xlabel=r["grouped_game"]["scale"])
    fig.suptitle(f"Mean absolute attribution · {n_global} distinct validation accounts")
    show(fig)
    display(
        Markdown(
            f"**{readable(ranked.index[0])}** has the largest average magnitude "
            f"({ranked.iloc[0]:.4g} on this scale). "
            "The ranking describes this model and reference population."
        )
    )

In [ ]:
if r:
    points = pd.DataFrame(primary["values"], columns=primary["columns"]).iloc[:n_global]
    originals = pd.DataFrame(r["feature_values"]).iloc[:n_global]
    leading = ranked.head(7).index
    fig, ax = plt.subplots(figsize=(10.6, 5))
    jitter = np.random.default_rng(31)
    for position, name in enumerate(reversed(leading)):
        if name in originals and pd.api.types.is_numeric_dtype(originals[name]):
            colour = originals[name].rank(pct=True).fillna(-0.1)
        else:
            colour = np.full(len(points), 0.5)
        artist = ax.scatter(
            points[name],
            position + jitter.uniform(-0.22, 0.22, len(points)),
            c=colour,
            cmap="coolwarm",
            vmin=0,
            vmax=1,
            s=19,
            alpha=0.68,
        )
    ax.axvline(0, color=grey, ls="--")
    ax.set(
        title=f"Signed account attributions · {n_global} development accounts",
        xlabel=f"Signed contribution · {primary['scale']}",
        yticks=range(len(leading)),
        yticklabels=[readable(x) for x in reversed(leading)],
    )
    if model == "churn":
        fig.colorbar(artist, ax=ax, label="Within-feature value rank · low to high")
    show(fig)

    cadence = pd.DataFrame(
        {name: row["mean_signed"] for name, row in r["global"]["cadences"].items()}
    ).loc[leading]
    display(pd.DataFrame(r["global"]["cadences"]).T[["accounts"]])
    fig, ax = plt.subplots(figsize=(10.5, 4.8))
    cadence.rename(index=readable).plot.bar(ax=ax, color=cadence_colours, rot=24)
    ax.axhline(0, color=grey, ls="--")
    ax.set(
        title=f"Mean signed contributions by cadence · {n_global} explained accounts",
        xlabel="Original feature or finite-game block",
        ylabel=primary["scale"],
    )
    ax.legend(title="Observed cadence")
    show(fig)

## Invoice exposure, cadence and the scope of a grouped game

The invoice-history block includes cadence, recent issuance, issue
size, frequency, recency and billing evidence. A scatterplot against
observed billed value illustrates its relationship with one of those
fields. The ordinate is an attribution for the entire block. Annual
and monthly support differs, and monthly repetition introduces
common forecast and outcome values. Small mixed-cadence error cohorts
are shown with their own denominators.

In [ ]:
if r:
    name = "episode_age_days" if model == "churn" else "Invoice history and cadence"
    field = "episode_age_days" if model == "churn" else "issued_gbp"
    key = "billing_frequency" if model == "churn" else "cadence"
    fig, ax = plt.subplots()
    if name in points:
        for level, group in originals.groupby(key):
            ax.scatter(
                group[field],
                points.loc[group.index, name],
                s=24,
                alpha=0.54,
                color=cadence_colours.get(level, grey),
                label=f"{level} · {len(group)}",
            )
    ax.axhline(0, color=grey, ls="--")
    ax.set(
        title=f"Attribution dependence on observed evidence · {n_global} accounts",
        xlabel="Episode age (days)" if model == "churn" else "Historical gross issued GBP",
        ylabel=primary["scale"],
    )
    if name in points:
        ax.legend(title="Cadence")
    show(fig)

    display(pd.DataFrame(r["validation_cadence"]).T)

## Error-conditioned financial contributions

Compare the mean absolute contribution of each block below the median
absolute error and in the upper error decile of the account-balanced
explained sample. These are descriptive cohorts with different
compositions. The observed errors supply group membership after the
contributions are computed. This comparison can highlight where to
examine the frozen model's failures.

In [ ]:
if r:
    outcomes = pd.DataFrame(r["explained_outcomes"]).iloc[:n_global]
    errors = (outcomes.prediction - outcomes.actual).abs()
    groups = {
        "At or below median error": errors <= errors.median(),
        "At or above 90th percentile": errors >= errors.quantile(0.90),
    }
    summary = pd.DataFrame({name: points.loc[mask].abs().mean() for name, mask in groups.items()})
    counts = pd.DataFrame(
        {
            name: {"Accounts": int(mask.sum()), "Median error GBP": errors.loc[mask].median()}
            for name, mask in groups.items()
        }
    ).T
    display(counts)
    fig, axis = plt.subplots(figsize=(9.5, 4.4))
    summary.plot.barh(ax=axis, color=[blue, warm])
    axis.set(
        title="Block attributions within development error cohorts",
        xlabel="Mean absolute contribution to clipped forecast (GBP)",
    )
    show(fig)

## Local cases and additive reconstruction

Case roles were fixed before attribution. Ties use retained observation
order and roles draw distinct accounts. The tables show the exact
model output and recorded outcome together. A waterfall starts from
the background mean, adds each signed term, and ends at the explained
prediction. The signed error remains separate from the decomposition.

The verified local roles include representative monthly and annual
forecasts, a low error, the largest absolute error, a mature zero
with a high forecast, high historical issuance and a clipped case.
The latter shows the raw-to-final difference without allocating a
separate artificial clipping adjustment.

In [ ]:
if r:
    case_rows = pd.DataFrame(r["cases"])
    case_rows["base"] = case_rows.position.map(lambda pos: primary["base"][pos])
    case_rows["base plus contributions"] = case_rows.position.map(
        lambda pos: primary["base"][pos] + sum(primary["values"][pos])
    )
    display(case_rows.drop(columns="position").round(9))
    if model == "churn" and "original" in r["tree_explanations"].get("raw", {}):
        margin = r["tree_explanations"]["raw"]["original"]
        display(
            pd.DataFrame(
                [
                    {
                        "Case": row["anonymous"],
                        "Log-odds base": margin["base"][row["position"]],
                        "Signed log-odds sum": sum(margin["values"][row["position"]]),
                        "Class-1 margin": row["raw"],
                        "Probability": row["final"],
                    }
                    for row in r["cases"]
                ]
            )
        )

In [ ]:
def waterfall(ax, case, evidence):
    pos = case["position"]
    values = pd.Series(evidence["values"][pos], index=evidence["columns"])
    order = values.abs().sort_values(ascending=False).index
    shown = values.loc[order[:5]]
    if len(order) > 5:
        shown = pd.concat(
            [shown, pd.Series({"All other mapped features": values.loc[order[5:]].sum()})]
        )
    labels = [readable(name) for name in shown.index]
    base = evidence["base"][pos]
    start = base + np.r_[0, np.cumsum(shown.to_numpy())[:-1]]
    end = start + shown.to_numpy()
    ax.barh(
        labels,
        np.abs(shown.to_numpy()),
        left=np.minimum(start, end),
        color=np.where(shown.to_numpy() >= 0, blue, warm),
    )
    ax.axvline(base, color=grey, ls="--", label="Background base")
    ax.axvline(case["final"], color="#17252c", ls=":", label="Exact prediction")
    ax.set(title=f"{case['anonymous']} · {case['label']}", xlabel=evidence["scale"])
    ax.invert_yaxis()
    ax.legend(loc="best", fontsize=8)


if r:
    for start in range(0, len(r["cases"]), 2):
        selected = r["cases"][start : start + 2]
        fig, axes = plt.subplots(1, len(selected), figsize=(13, 4.8), squeeze=False)
        for axis, case in zip(axes[0], selected, strict=True):
            waterfall(axis, case, primary)
        show(fig)

## Supported partial dependence and individual conditional expectation

An ICE line evaluates $f(x_{-j}^{(i)},z)$ for one case at a supported
value $z$. The displayed PDP is the mean over supported validation
accounts at each grid point. The set of supported accounts can change.
A training-support check uses observed cadence and distance in a set of
relevant companion variables. The predeclared local radius is 0.30
training IQR units under maximum coordinate distance. The display requires
at least five accounts for a global point.

For two supported ICE predictions of 40 and 60, their descriptive PDP
value is 50. This changes supplied predictors in the model. Real-world
interventions would require separate evidence about feasible changes.

In [ ]:
if r:
    for name, entry in r["sensitivity"]["curves"].items():
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
        grid = entry["grid"]
        field_is_category = isinstance(grid[0], str)
        axes[0].plot(
            grid,
            entry["pdp"],
            "o" if field_is_category else "o-",
            color="#17252c",
            label="Supported PDP mean",
            lw=2.3,
        )
        for case, values in entry["ice"].items():
            if any(value is not None for value in values):
                short = next(row["anonymous"] for row in r["cases"] if row["label"] == case)
                axes[0].plot(
                    grid, values, "." if field_is_category else ".--", alpha=0.64, label=short
                )
        axes[0].set(
            title="Partial dependence and selected ICE", ylabel=units, xlabel=readable(name)
        )
        axes[0].legend(fontsize=8)
        axes[1].bar(range(len(grid)), entry["support"], color=blue)
        axes[1].axhline(5, color=warm, ls="--", label="PDP support floor")
        axes[1].set(
            title="Support at each fixed grid point",
            xlabel=readable(name),
            ylabel="Supported accounts",
            xticks=range(len(grid)),
            xticklabels=grid,
            ylim=(0, entry["candidate_accounts"] + 6),
        )
        axes[1].legend(fontsize=8)
        fig.suptitle(
            f"{readable(name).capitalize()} · {entry['candidate_accounts']} sampled accounts"
        )
        show(fig)
        display(
            Markdown(
                f"**{sum(value is not None for value in entry['pdp'])} of {len(grid)}** points "
                f"meet the support floor for {readable(name)}. Interpret changes with the "
                "support counts, since accounts may differ across points."
            )
        )

## Observed-support interaction diagnostics

For frozen pairs, display jointly supported predictions on the fixed
grid. A two-by-two interaction contrast is
$f_{11}-f_{10}-f_{01}+f_{00}$ on the same accounts. At least five
accounts must support all corners, so four individually supported
marginal means cannot substitute for this check. Blank values mark
unsupported cells. This screens out misleading pairwise inferences.

In [ ]:
if r:
    for entry in r["sensitivity"]["interactions"]:
        values = np.asarray(entry["surface"], dtype=float)
        counts = np.asarray(entry["support"])
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
        for data, title, palette in (
            (counts, "Joint support", "Blues"),
            (values, "Mean prediction where supported", "viridis"),
        ):
            ax = axes[0] if title == "Joint support" else axes[1]
            image = ax.imshow(data, cmap=palette, aspect="auto")
            for (i, j), value in np.ndenumerate(data):
                ax.text(
                    j,
                    i,
                    f"{value:.3g}" if np.isfinite(value) else "—",
                    ha="center",
                    va="center",
                    color="black",
                    bbox={"facecolor": "white", "alpha": 0.76, "pad": 2, "edgecolor": "none"},
                )
            ax.grid(False)
            x, y = entry["features"]
            ax.set(
                title=title,
                xlabel=readable(x),
                ylabel=readable(y),
                xticks=range(len(entry["x"])),
                xticklabels=entry["x"],
                yticks=range(len(entry["y"])),
                yticklabels=entry["y"],
            )
            fig.colorbar(image, ax=ax, label="Accounts" if title == "Joint support" else units)
        fig.suptitle("Prespecified pair · finite development support, five-account minimum")
        show(fig)
        matched = pd.DataFrame(entry["matched_contrasts"])
        if not matched.empty:
            display(matched.rename(columns={"matched": "Accounts with all four corners"}))
        supported = sum(row["effect"] is not None for row in entry["matched_contrasts"])
        display(Markdown(f"**Matched contrasts meeting support:** {supported}."))

## Stability, replay and background sensitivity

Identical background and model inputs are replayed first. Then each
frozen alternate background is compared with the primary signed
decomposition. Global rank summaries use tied-rank Spearman and
top-set Jaccard overlap. Account resampling uses 100 bootstrap draws
from one explained row per account. The underlying estimator is held
fixed. Local variation across backgrounds measures explanation
dependence on the reference population. Predictive error uncertainty
and model generalisation require separate evidence.

Rank comparisons use mean absolute attribution on 96 validation
accounts. Background comparisons hold those accounts fixed. Each
bootstrap resamples 96 account representatives with replacement
at fixed background. The reported maximum reference-driven shift
is across all explained rows and all contribution columns.

In [ ]:
if r:
    stability = r["stability"]
    summary = pd.DataFrame(stability["summary"]).T
    display(summary)
    bg = pd.DataFrame(
        [
            {
                "Seed": row["background"]["seed"],
                "Accounts": row["background"]["accounts"],
                "Billing mix": row["background"]["cadence"],
                "Available": row["available"],
                "Rank correlation": row.get("global", {}).get("spearman"),
                "Base shift": row.get("base_shift"),
                "Largest local movement": row.get("maximum_local_abs_change"),
            }
            for row in stability["background_comparisons"]
        ]
    )
    display(bg)
    sampling = pd.DataFrame(stability["sampling"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    for name, hue in (("spearman", blue), ("top_jaccard", warm)):
        axes[0].hist(
            sampling[name].dropna(),
            bins=np.linspace(0, 1, 21),
            histtype="step",
            lw=2,
            color=hue,
            label=readable(name),
        )
    axes[0].set(title="100 account bootstrap draws", xlabel="Ranking agreement", ylabel="Draws")
    axes[0].legend()
    series = bg["Seed"].astype(str) + " / " + bg["Accounts"].astype(str)
    axes[1].bar(series, bg["Largest local movement"], color=warm)
    axes[1].set(
        title="Largest local attribution movement",
        xlabel="Background seed / accounts",
        ylabel=primary["scale"],
    )
    show(fig)

    demonstration = r["cases"][1]
    small = pd.DataFrame(
        stability["local_values"][demonstration["label"]], columns=primary["columns"]
    )[ranked.head(4).index]
    fig, ax = plt.subplots()
    small.rename(columns=readable).plot(marker="o", ax=ax)
    ax.set(
        title=f"{demonstration['anonymous']} · contribution sensitivity",
        xlabel="Primary background then available alternatives",
        ylabel=primary["scale"],
        xticks=range(len(small)),
    )
    ax.legend(fontsize=8)
    show(fig)
    display(
        Markdown(
            f"Identical-background replay maximum absolute change "
            f"**{stability['replay_max_abs_delta']:.3g}** on this scale. "
            "Alternative reference cohorts can change the local baseline and contributions."
        )
    )
    scale_floor = primary["reconciliation"]["tolerance"]
    local_summary = []
    for case in r["cases"]:
        repeated = np.asarray(stability["local_values"][case["label"]])
        spread = np.ptp(repeated, axis=0)
        index = int(np.argmax(spread))
        above = repeated[:, index] > scale_floor
        below = repeated[:, index] < -scale_floor
        local_summary.append(
            {
                "Case": case["anonymous"],
                "Largest-spread term": readable(primary["columns"][index]),
                "Signed span": float(spread[index]),
                "Sign crosses zero": bool(above.any() and below.any()),
                "Near-zero run present": bool((~(above | below)).any()),
                "Backgrounds": len(repeated),
            }
        )
    display(pd.DataFrame(local_summary))

## Feature correlation and interpretation limits

Spearman correlation describes monotone associations on the validation
rows. It is descriptive, and repeated account observations contribute
more than one row in this view. Correlated invoice predictors can share
attribution and offer substitutes during singleton permutation. Tree
and grouped callable explanations each use a specified background;
correlated hybrids can occur even when related inputs are moved jointly.

In [ ]:
if r:
    selected = (
        [
            "episode_age_days",
            "monthly_recurring_gbp",
            "issued_amount_gbp",
            "issued_invoice_count",
            "days_since_last_invoice",
            "latest_sessions",
            "mean_sessions",
        ]
        if model == "churn"
        else [
            "issued_gbp",
            "recent_gbp",
            "invoice_count",
            "recency_days",
            "mean_invoice_gbp",
            "snapshot_gbp",
            "tenure_days",
        ]
    )
    corr = pd.DataFrame(r["rank_correlations"]).loc[selected, selected]
    fig, ax = plt.subplots(figsize=(8.5, 6.4))
    image = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
    labels = [readable(name) for name in corr.index]
    ax.set(
        title=f"Validation rank correlations · {len(dist)} historical rows",
        xticks=range(len(labels)),
        xticklabels=labels,
        yticks=range(len(labels)),
        yticklabels=labels,
    )
    ax.tick_params(axis="x", labelrotation=42)
    ax.grid(False)
    fig.colorbar(image, ax=ax, label="Spearman correlation")
    show(fig)

## Scientific assessment

Invoice history and cadence dominate both the finite grouped
attribution and the matched-account MAE permutations. That joint
result cannot isolate an independent causal effect of cadence or a
single invoice metric. Account lifecycle has a smaller measured
contribution. Keeping observation dates fixed leaves calendar
importance unavailable under the permutation design, even though
calendar receives contribution in the separate additive game.
A large local miss remains a model error after its components
reconcile. Annual and mixed cohort errors are higher in this
fictional development evidence. Commercial value, causal use
and real-customer generalisation require separate evidence.

## Reproduction

The notebook reads the frozen protocol, manifest and selected report
from the Phase 09 workspace. The research runner validates model and
input fingerprints prior to masked predictions and writes all outputs
outside Git. Execute the repository source through a temporary review
copy in a fresh kernel. [Notebook instructions](../README.md).